In [1]:
import json
import pandas as pd
import chardet
from pprint import pprint

In [137]:
with open('reviews.json', 'rb') as f:
    rawdata = f.read()
    result = chardet.detect(rawdata)
    print(result['encoding'])


MacRoman


In [138]:
with open('reviews.json', 'r', encoding='mac_roman') as f:
    reviews = json.load(f)

pprint(reviews[0]['fields'])

{'content': "(As I'm writing this review, Darth Vader's theme music begins to "
            'build in my mind...)\r\n'
            '\r\n'
            'Well, it actually has a title, what the Darth Vader theme. And '
            'that title is "The Imperial March", composed by the great John '
            'Williams, whom, as many of you may already know, also composed '
            'the theme music for "Jaws" - that legendary score simply titled, '
            '"Main Title (Theme From Jaws)".\r\n'
            '\r\n'
            "Now, with that lil' bit of trivia aside, let us procede with the "
            'fabled film currently under review: Star Wars. It had been at a '
            'drive-in theater in some small Illinois town or other where my '
            'mother, my older brother, and I had spent our weekly "Movie Date '
            'Night" watching this George Lucas directed cult masterpiece from '
            'our car in the parking lot. On the huge outdoor screen, the film '
  

# 1. 필요한 정보만 가져와 df로 바꾸기

movie_id, username, content, rating만 가져오기

In [139]:
columns = ['movie_id', 'user_name', 'content', 'rating']

data = [
]

for dict in reviews:
    fields = dict['fields']
    temp_lst = []
    temp_lst.append(fields['movie_id'])
    temp_lst.append(fields['username'])
    temp_lst.append(fields['content'])
    temp_lst.append(fields['rating'])
    data.append(temp_lst)

review_df = pd.DataFrame(data, columns=columns)

review_df.head(20)

,movie_id,user_name,content,rating
0,11,Cat Ellington,"(As I'm writing this review, Darth Vader's the...",NaN
1,11,John Chard,A long time ago in a childhood not too far awa...,10.0
2,11,gastyny,Star Wars (1977) is a true masterpiece of cine...,10.0
3,11,r96sk,A quality start to the franchise.\r\n\r\nI say...,9.0
4,11,GenerationofSwine,Everyone and their mother is going to write re...,10.0
5,11,CinemaSerf,Thinking back to the films that define my gene...,9.0
6,12,Dave09,One of the best animated films I have ever see...,NaN
7,12,r96sk,Utterly stunning.\r\n\r\nThere isn't anything ...,9.0
8,12,John,Awesome ocean visuals and fun story do a good ...,10.0
9,12,CinemaSerf,"""Nemo"" is your typically adventurous and curio...",7.0


## 이름에서 특수 문자, 공백 없애주기

In [140]:
def remove(text):
    return ''.join(char for char in text if char.isalnum())

review_df['user_name'] = review_df['user_name'].apply(remove)
review_df.head(20)

,movie_id,user_name,content,rating
0,11,CatEllington,"(As I'm writing this review, Darth Vader's the...",NaN
1,11,JohnChard,A long time ago in a childhood not too far awa...,10.0
2,11,gastyny,Star Wars (1977) is a true masterpiece of cine...,10.0
3,11,r96sk,A quality start to the franchise.\r\n\r\nI say...,9.0
4,11,GenerationofSwine,Everyone and their mother is going to write re...,10.0
5,11,CinemaSerf,Thinking back to the films that define my gene...,9.0
6,12,Dave09,One of the best animated films I have ever see...,NaN
7,12,r96sk,Utterly stunning.\r\n\r\nThere isn't anything ...,9.0
8,12,John,Awesome ocean visuals and fun story do a good ...,10.0
9,12,CinemaSerf,"""Nemo"" is your typically adventurous and curio...",7.0


In [141]:
review_df.isnull().sum()

movie_id       0
user_name      0
content        0
rating       169
dtype: int64

#  2. rating이 없는 리뷰에 대해 텍스트 감성분석

In [7]:
import re
from functools import reduce

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud, STOPWORDS

In [36]:
nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [142]:
review_df['content']

0       (As I'm writing this review, Darth Vader's the...
1       A long time ago in a childhood not too far awa...
2       Star Wars (1977) is a true masterpiece of cine...
3       A quality start to the franchise.\r\n\r\nI say...
4       Everyone and their mother is going to write re...
                              ...                        
1605    There are times in our lives when nearly all o...
1606    Brimming with thoughtful themes and stunning a...
1607    "Roz" is pre-programmed to help. Whatever it i...
1608    FULL SPOILER-FREE REVIEW @ https://movieswetex...
1609            Dreamworks at its best!!!! üòçüò≠‚ù§Ô∏è
Name: content, Length: 1610, dtype: object

In [143]:
null_indices = review_df[review_df['rating'].isnull()].index            # rating이 null인 인덱스
null_contents = review_df.loc[review_df['rating'].isnull(), 'content']  # 걔네의 리뷰

In [144]:
len(null_indices)

169

In [145]:
stopWords = set(stopwords.words("english"))
lemma = WordNetLemmatizer()

## 필요한 단어만 추출

In [146]:
words = []  

for title in null_contents:
    EnWords = re.sub(r"[^a-zA-Z]+", " ", str(title))    
    EnWordsToken = word_tokenize(EnWords.lower())
    EnWordsTokenStop = [w for w in EnWordsToken if w not in stopWords]
    EnWordsTokenStopLemma = [lemma.lemmatize(w) for w in EnWordsTokenStop]
    words.append(EnWordsTokenStopLemma)

len(words)

169

## 감성분석

In [147]:
from transformers import pipeline
import torch

In [148]:
sentiment_anaylysis = pipeline('sentiment-analysis')

def senti(lst):
    text = ' '.join(lst)
    result = sentiment_anaylysis(text, truncation=True)[0]

    return 1 if result['label'] == 'POSITIVE' else 0

result = []

for word in words:
    result.append(senti(word))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


# 3. 데이터 프레임 재조정

평점이 6점 이상이면 like, 미만이면 0

In [149]:
review_df['like'] = review_df['rating'].apply(lambda x: 1 if x >= 6 else 0)
review_df.head(20)

,movie_id,user_name,content,rating,like
0,11,CatEllington,"(As I'm writing this review, Darth Vader's the...",NaN,0
1,11,JohnChard,A long time ago in a childhood not too far awa...,10.0,1
2,11,gastyny,Star Wars (1977) is a true masterpiece of cine...,10.0,1
3,11,r96sk,A quality start to the franchise.\r\n\r\nI say...,9.0,1
4,11,GenerationofSwine,Everyone and their mother is going to write re...,10.0,1
5,11,CinemaSerf,Thinking back to the films that define my gene...,9.0,1
6,12,Dave09,One of the best animated films I have ever see...,NaN,0
7,12,r96sk,Utterly stunning.\r\n\r\nThere isn't anything ...,9.0,1
8,12,John,Awesome ocean visuals and fun story do a good ...,10.0,1
9,12,CinemaSerf,"""Nemo"" is your typically adventurous and curio...",7.0,1


```python
null_indices  # rating이 null인 데이터의 인덱스

result        # 그 데이터들의 감성분석 결과
```

In [150]:
for row_idx in null_indices:
    review_df.iloc[row_idx, 4] = result[0]
    result.pop(0)

In [151]:
review_df.isnull().sum()

movie_id       0
user_name      0
content        0
rating       169
like           0
dtype: int64

In [152]:
df = review_df[['movie_id', 'user_name', 'like']]
print(df['like'].value_counts())
df

like
1    1447
0     163
Name: count, dtype: int64


,movie_id,user_name,like
0,11,CatEllington,1
1,11,JohnChard,1
2,11,gastyny,1
3,11,r96sk,1
4,11,GenerationofSwine,1
...,...,...,...
1605,1184918,BrentMarchant,1
1606,1184918,goodfilm,1
1607,1184918,CinemaSerf,1
1608,1184918,ManuelSoBento,1


In [ ]:
df.to_json('sentiment.json', orient='records', indent=4)
df.to_csv('sentiment.csv', index=False)